In [1]:

import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Create output folder
# -------------------------
os.makedirs("charts", exist_ok=True)

# -------------------------
# Load Data
# -------------------------
ledger = pd.read_csv("ledger.csv")
gateway = pd.read_csv("gateway_export.csv")
merchants = pd.read_csv("merchants.csv")

ledger["transaction_time"] = pd.to_datetime(ledger["transaction_time"])

# -------------------------
# Merge Merchant Details
# -------------------------
ledger = ledger.merge(
    merchants,
    on="merchant_id",
    how="left"
)

# -------------------------
# KPI Calculations
# -------------------------

total_gmv = ledger["amount_inr"].sum()

success_rate = (
    (ledger["status"] == "captured").sum()
    / len(ledger)
) * 100

chargeback_ratio = (
    (ledger["status"] == "chargeback").sum()
    / len(ledger)
) * 100

merged = ledger.merge(
    gateway,
    on="transaction_id",
    suffixes=("_ledger", "_gateway")
)

matched = merged[
    (merged["amount_inr_ledger"] == merged["amount_inr_gateway"])
    &
    (merged["status_ledger"] == merged["status_gateway"])
]

match_rate = len(matched) / len(ledger) * 100

print("=" * 40)
print("PAYTM ANALYTICS DASHBOARD")
print("=" * 40)
print(f"Total GMV          : ₹{total_gmv:,.2f}")
print(f"Success Rate       : {success_rate:.2f}%")
print(f"Chargeback Ratio   : {chargeback_ratio:.2f}%")
print(f"Reconciliation Rate: {match_rate:.2f}%")

# -------------------------
# Daily GMV
# -------------------------

daily = (
    ledger
    .groupby(ledger["transaction_time"].dt.date)
    ["amount_inr"]
    .sum()
)

plt.figure(figsize=(10,5))
plt.plot(daily.index, daily.values, marker="o")
plt.title("Daily GMV")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("charts/daily_gmv.png")
plt.close()

# -------------------------
# Daily Chargebacks
# -------------------------

chargebacks = (
    ledger[ledger["status"]=="chargeback"]
    .groupby(ledger["transaction_time"].dt.date)
    .size()
)

plt.figure(figsize=(10,5))
plt.plot(chargebacks.index, chargebacks.values, marker="o")
plt.title("Daily Chargeback Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("charts/daily_chargebacks.png")
plt.close()

# -------------------------
# GMV by Payment Method
# -------------------------

payment = (
    ledger
    .groupby("payment_method")
    ["amount_inr"]
    .sum()
)

plt.figure(figsize=(7,5))
payment.plot(kind="bar")
plt.ylabel("GMV")
plt.title("GMV by Payment Method")
plt.tight_layout()
plt.savefig("charts/payment_method.png")
plt.close()

# -------------------------
# GMV by Category
# -------------------------

category = (
    ledger
    .groupby("category")
    ["amount_inr"]
    .sum()
)

plt.figure(figsize=(8,5))
category.plot(kind="bar")
plt.ylabel("GMV")
plt.title("GMV by Category")
plt.tight_layout()
plt.savefig("charts/category.png")
plt.close()

# -------------------------
# Top 10 Merchants
# -------------------------

top = (
    ledger
    .groupby("merchant_name")
    .agg(
        Transactions=("transaction_id","count"),
        GMV=("amount_inr","sum"),
        Chargebacks=("status",lambda x:(x=="chargeback").sum())
    )
)

top["Chargeback Ratio (%)"] = (
    top["Chargebacks"]
    / top["Transactions"]
    *100
)

top = top.sort_values(
    by="Transactions",
    ascending=False
).head(10)

fig, ax = plt.subplots(figsize=(12,5))
ax.axis("off")

table = ax.table(
    cellText=top.round(2).values,
    colLabels=top.columns,
    rowLabels=top.index,
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2,1.4)

plt.savefig("charts/top10_merchants.png")
plt.close()

print("\nDashboard images saved in charts/ folder.")

PAYTM ANALYTICS DASHBOARD
Total GMV          : ₹382,603.00
Success Rate       : 85.56%
Chargeback Ratio   : 5.12%
Reconciliation Rate: 90.49%

Dashboard images saved in charts/ folder.
